In [50]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
%autoreload 2

In [52]:
import numpy as np
import pandas as pd
import yfinance as yf


In [53]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [54]:
from services.market_service import load_live_chain, filter_chain_with_stats, parse_tickers

tickers = ['NVDA']
r = 0.05
q = 0.0
spread_limit = 0.05

# Raw chain — no filtering applied
raw_df = load_live_chain(tickers=parse_tickers(tickers))
print(f"Raw contracts: {len(raw_df)}")

# Apply all filters (same logic as the app)
filtered_df, filter_stats = filter_chain_with_stats(
    raw_df,
    spread_limit=spread_limit,
    r=r,
    q=q,
    tickers=parse_tickers(tickers),
    option_types=("call", "put"),
    min_volume=1,
    min_open_interest=0,
    max_maturity=2.0,
    # No max_contracts cap — see all contracts that pass quality filters
)
print(f"Filtered contracts: {len(filtered_df)}")
print("\nFilter breakdown:")
for reason, count in filter_stats.items():
    print(f"  {reason}: {count}")

filtered_df.head()

pulling...NVDA...
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness'],
      dtype='object')
Raw contracts: 4044
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness', 'spot_over_strike', 'atm_distance',
       'contract_id'],
      dtype='object')
Filtered contracts: 792

Filter breakdown:
  Mid price ≤ 0.001: 46
  Rel. spread ≥ 5%: 1209
  Moneyness outside [0.8, 1.2]: 1935
  Arbitrage violation: 1
  Maturity > 2.0y: 61


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,spot,ticker,ExerciseStyle,T,mid_price,rel_spread,moneyness,spot_over_strike,atm_distance,contract_id
0,NVDA260508C00170000,2026-05-06 19:59:35+00:00,170.0,38.30,37.70,39.35,10.41,37.325207,106.0,743.0,...,207.830002,NVDA,american,0.00275,38.525,0.042829,0.817976,1.222529,0.200922,NVDA260508C00170000
1,NVDA260508C00177500,2026-05-06 19:57:44+00:00,177.5,30.60,30.20,31.15,11.60,61.052630,375.0,193.0,...,207.830002,NVDA,american,0.00275,30.675,0.030970,0.854063,1.170873,0.157750,NVDA260508C00177500
2,NVDA260508C00180000,2026-05-06 19:57:38+00:00,180.0,28.09,27.70,28.40,11.59,70.242424,335.0,3894.0,...,207.830002,NVDA,american,0.00275,28.050,0.024955,0.866092,1.154611,0.143764,NVDA260508C00180000
3,NVDA260508C00182500,2026-05-06 19:56:42+00:00,182.5,25.71,25.25,26.25,11.52,81.183940,29.0,341.0,...,207.830002,NVDA,american,0.00275,25.750,0.038835,0.878122,1.138795,0.129970,NVDA260508C00182500
4,NVDA260508C00190000,2026-05-06 19:59:51+00:00,190.0,18.09,17.80,18.10,10.58,140.878830,3881.0,10398.0,...,207.830002,NVDA,american,0.00275,17.950,0.016713,0.914209,1.093842,0.089696,NVDA260508C00190000


## Data-Driven Heston Bounds — Dry Run (NVDA)

Estimate initial guess and tight search bounds for `(v0, kappa, theta, sigma, rho)`
directly from the market implied-vol surface, following:

| Param | Source |
|-------|--------|
| `v0`    | ATM IV² at shortest liquid maturity |
| `theta` | ATM IV² at longest liquid maturity  |
| `sigma` | vol-of-vol from smile curvature: `σ ≈ √(8c)` |
| `rho`   | correlation from ATM slope: `ρ ≈ 2b/σ` |
| `kappa` | mean reversion from term-structure slope |

In [55]:
import warnings
warnings.filterwarnings("ignore")
import sys
import os
sys.path.append(os.path.abspath(".."))
import pandas as pd

In [56]:
from services.market_service import load_live_chain, parse_tickers
from analytics.schema import ensure_option_frame
from data.option_filters import apply_filters
from calibration.data_driven_bounds import compute_data_driven_bounds
from calibration.implied_vol import implied_volatility

In [57]:
# --- Load data ---------------------------------------------------------------
# Use sample Excel data so the dry run is reproducible regardless of market hours.
# The live filtered_df (from cell above) can be substituted if markets are open.

tickers = ['NVDA']
r = 0.05
q = 0.0
spread_limit = 0.05

# Raw chain — no filtering applied
df_raw = load_live_chain(tickers=parse_tickers(tickers))

df = ensure_option_frame(df_raw)

#Apply filteres
filtered_df , stats = apply_filters(
    df,
    spread_limit=0.10,
    r=0.05, q=0.0,
    moneyness_lo=0.7, moneyness_hi=1.3,
    min_volume=0, min_open_interest=0,
    keep_positive_time=True,        # live data
)

print(filtered_df.columns)
print(len(df_raw))
print(len(df))
print(len(filtered_df))

# Drop effectively-expired rows; keep only contracts with meaningful time value
filtered_df = filtered_df[filtered_df["T"] > 0.01].copy()

filtered_df.head()


pulling...NVDA...
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness'],
      dtype='object')
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness', 'spot_over_strike', 'atm_distance',
       'contract_id'],
      dtype='object')
Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturit

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,...,spot,ticker,ExerciseStyle,T,mid_price,rel_spread,moneyness,spot_over_strike,atm_distance,contract_id
36,NVDA260511C00177500,2026-05-04 19:57:45+00:00,177.5,21.15,29.85,31.25,0.000000,0.00000,76.0,32.0,...,207.830002,NVDA,american,0.010969,30.550,0.045827,0.854063,1.170873,0.157750,NVDA260511C00177500
37,NVDA260511C00180000,2026-05-06 15:40:37+00:00,180.0,25.81,27.45,28.60,9.119999,54.64349,16.0,68.0,...,207.830002,NVDA,american,0.010969,28.025,0.041035,0.866092,1.154611,0.143764,NVDA260511C00180000
38,NVDA260511C00182500,2026-05-06 17:05:02+00:00,182.5,22.44,24.95,26.30,6.740001,42.92994,3.0,34.0,...,207.830002,NVDA,american,0.010969,25.625,0.052683,0.878122,1.138795,0.129970,NVDA260511C00182500
39,NVDA260511C00185000,2026-05-06 19:57:36+00:00,185.0,23.01,22.50,23.85,9.910000,75.64885,19.0,78.0,...,207.830002,NVDA,american,0.010969,23.175,0.058252,0.890151,1.123405,0.116365,NVDA260511C00185000
40,NVDA260511C00187500,2026-05-06 19:57:02+00:00,187.5,20.57,20.05,21.30,10.520000,104.67661,13.0,30.0,...,207.830002,NVDA,american,0.010969,20.675,0.060459,0.902180,1.108427,0.102942,NVDA260511C00187500


In [58]:
result = compute_data_driven_bounds(
    filtered_df,
    r=0.05,
    q=0.0,
    atm_threshold=0.10,   # |log(K/S)| < 10% for ATM IV estimation
    fit_threshold=0.15,   # |log(K/S)| < 15% for skew/curvature regression
    min_contracts=5
)

In [59]:

#sample_path = "../nvda_vol.xlsx"
#df_raw = pd.read_excel(sample_path)
df = ensure_option_frame(df_raw)

sample_df, stats = apply_filters(
    df,
    spread_limit=0.10,
    r=0.05, q=0.0,
    moneyness_lo=0.7, moneyness_hi=1.3,
    min_volume=0, min_open_interest=0,
    keep_positive_time=False,        # stored T, not live
)
# Drop effectively-expired rows; keep only contracts with meaningful time value
sample_df = sample_df[sample_df["T"] > 0.01].copy()

print(f"Sample: {len(sample_df)} contracts  |  {sample_df['T'].nunique()} unique maturities  |  "
      f"T ∈ [{sample_df['T'].min():.3f}, {sample_df['T'].max():.3f}] yr")

# --- Compute data-driven bounds ----------------------------------------------
result = compute_data_driven_bounds(
    sample_df,
    r=0.05,
    q=0.0,
    atm_threshold=0.10,   # |log(K/S)| < 10% for ATM IV estimation
    fit_threshold=0.15,   # |log(K/S)| < 15% for skew/curvature regression
    min_contracts=5,
)

ig   = result["initial_guess"]
bnds = result["bounds"]
diag = result["diagnostics"]

# --- Print results -----------------------------------------------------------
print("\n" + "=" * 65)
print("  NVDA — Data-Driven Heston Initial Guess & Bounds")
print("=" * 65)

if "warning" in diag:
    print(f"  WARNING: {diag['warning']}")
    print("  (Showing static fallback defaults)")
else:
    labels = [
        "v0     initial variance",
        "kappa  mean reversion  ",
        "theta  long-run variance",
        "sigma  vol-of-vol      ",
        "rho    correlation     ",
    ]
    for label, guess, (lo, hi) in zip(labels, ig, bnds):
        print(f"  {label}  guess = {guess:+.4f}   bound = [{lo:+.4f}, {hi:+.4f}]")

    print()
    print("  ── Surface diagnostics ──────────────────────────────────")
    print(f"  Short maturity  T_short      = {diag['T_short']:.4f} yr  "
          f"({diag['T_short']*365:.1f} days)")
    print(f"  Long  maturity  T_long       = {diag['T_long']:.4f} yr  "
          f"({diag['T_long']*365:.1f} days)")
    print(f"  ATM IV  (T_short)            = {diag['sigma_atm_short']:.4f}  "
          f"({diag['sigma_atm_short']*100:.2f} %)")
    print(f"  ATM IV  (T_long )            = {diag['sigma_atm_long']:.4f}  "
          f"({diag['sigma_atm_long']*100:.2f} %)")
    print(f"  Term-structure slope         = {diag['slope_T']:+.4f} vol / yr")
    print(f"  Smile slope  b  (skew)       = {diag['smile_slope_b']:+.4f}")
    print(f"  Smile curvature c            = {diag['smile_curvature_c']:+.4f}")
    print(f"  Liquid maturities  ({diag['n_liquid_maturities']:2d}):      "
          f"{[round(t, 3) for t in diag['liquid_maturities']]}")

print("=" * 65)

Index(['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask',
       'change', 'percentChange', 'volume', 'openInterest',
       'impliedVolatility', 'inTheMoney', 'contractSize', 'currency', 'type',
       'maturity', 'spot', 'ticker', 'ExerciseStyle', 'T', 'mid_price',
       'rel_spread', 'moneyness', 'spot_over_strike', 'atm_distance',
       'contract_id'],
      dtype='object')
Sample: 1243 contracts  |  22 unique maturities  |  T ∈ [0.011, 2.611] yr

  NVDA — Data-Driven Heston Initial Guess & Bounds
  v0     initial variance  guess = +0.1612   bound = [+0.1128, +0.2095]
  kappa  mean reversion    guess = +0.3846   bound = [+0.5000, +1.1538]
  theta  long-run variance  guess = +0.2170   bound = [+0.1085, +0.4340]
  sigma  vol-of-vol        guess = +2.0000   bound = [+1.0000, +2.0000]
  rho    correlation       guess = -0.6303   bound = [-0.8303, -0.4303]

  ── Surface diagnostics ──────────────────────────────────
  Short maturity  T_short      = 0.0110 yr  (4.0

In [60]:
print(ig)

[0.1611715941478722, 0.38461539563284436, 0.21698780553825298, 2.0, -0.6303072998206933]


## Historical Bounds — Physical Measure (NVDA, 2y / 3-month RV window)

Estimates Heston parameters from 2 years of daily closing prices under the **physical measure P**.

| Parameter | Estimator |
|-----------|-----------|
| `v0`    | Most recent 3-month (63-day) rolling realized variance |
| `theta` | Long-run mean of the rolling RV series over full 2-year history |
| `kappa` | AR(1) on RV series: `v[t+1] = a + b·v[t]`  →  `κ = (1 − b) / dt` |
| `sigma` | Normalized AR(1) residuals: `σ = std(ε / √(v[t]·dt))` |
| `rho`   | `corr(daily log-return,  Δ rolling RV)` |

> **Physical vs risk-neutral:** these P-measure estimates validate the *sign* and *order of magnitude* of the smile-derived bounds. Exact values differ — physical `rho` is typically weaker (closer to zero) than risk-neutral `rho`.

In [61]:
from calibration.historical_bounds import compute_historical_bounds

# ── Fetch 2y of NVDA daily prices and estimate parameters ────────────────────
hist_result = compute_historical_bounds("NVDA", period="2y", v0_window=63)

h_ig   = hist_result["initial_guess"]
h_bnds = hist_result["bounds"]
h_diag = hist_result["diagnostics"]

# ── Side-by-side comparison with smile-derived bounds ────────────────────────
PARAM_LABELS = [
    "v0     initial variance ",
    "kappa  mean reversion   ",
    "theta  long-run variance",
    "sigma  vol-of-vol       ",
    "rho    correlation      ",
]

print("=" * 90)
print(f"  {'NVDA — Parameter Comparison':^86}")
print(f"  {'Smile-derived (Q measure)':^40}  {'Historical (P measure, 3-month RV)':^40}")
print("=" * 90)
print(f"  {'Parameter':<25} {'guess':>8}  {'bound':^22}  {'guess':>8}  {'bound':^22}")
print("-" * 90)

for label, s_g, (s_lo, s_hi), h_g, (h_lo, h_hi) in zip(
    PARAM_LABELS, ig, bnds, h_ig, h_bnds
):
    print(
        f"  {label:<25} {s_g:+8.4f}  [{s_lo:+.4f}, {s_hi:+.4f}]"
        f"  {h_g:+8.4f}  [{h_lo:+.4f}, {h_hi:+.4f}]"
    )

print("=" * 90)
print()

# ── Historical diagnostics ────────────────────────────────────────────────────
# half-life in trading days = ln(2) / (1 - b)
b = h_diag["ar1_b"]
half_life_days = 0.693 / max(1.0 - b, 1e-8)

print("  ── Historical diagnostics ───────────────────────────────────────────")
print(f"  History          : {h_diag['history_start']}  →  {h_diag['history_end']}")
print(f"  Price obs        : {h_diag['n_price_obs']}  |  Return obs: {h_diag['n_return_obs']}")
print(f"  RV window        : {h_diag['v0_window_days']} trading days (~3 months)")
print(f"  RV (latest)      : {h_diag['rv_latest']:.4f}  "
      f"  → annualized vol = {h_diag['rv_latest']**0.5 * 100:.2f}%")
print(f"  RV (mean / std)  : {h_diag['rv_mean']:.4f}  /  {h_diag['rv_std']:.4f}")
print(f"  RV (min  / max)  : {h_diag['rv_min']:.4f}  /  {h_diag['rv_max']:.4f}")
print(f"  AR(1) b          : {b:.6f}  "
      f"(half-life ≈ {half_life_days:.0f} trading days  /  {half_life_days/21:.1f} months)")
print(f"  AR(1) θ implied  : {h_diag['ar1_implied_theta']:.4f}  "
      f"  → vol = {h_diag['ar1_implied_theta']**0.5 * 100:.2f}%")
print(f"  Feller satisfied : {h_diag['feller_satisfied']}  "
      f"  (2κθ = {2*h_ig[1]*h_ig[2]:.4f}  vs  σ² = {h_ig[3]**2:.4f})")
print("=" * 90)

                               NVDA — Parameter Comparison                              
         Smile-derived (Q measure)             Historical (P measure, 3-month RV)   
  Parameter                    guess          bound              guess          bound         
------------------------------------------------------------------------------------------
  v0     initial variance    +0.1612  [+0.1128, +0.2095]   +0.1514  [+0.1060, +0.1969]
  kappa  mean reversion      +0.3846  [+0.5000, +1.1538]   +1.2563  [+0.6281, +3.7688]
  theta  long-run variance   +0.2170  [+0.1085, +0.4340]   +0.2409  [+0.1205, +0.4818]
  sigma  vol-of-vol          +2.0000  [+1.0000, +2.0000]   +0.4341  [+0.2170, +0.8682]
  rho    correlation         -0.6303  [-0.8303, -0.4303]   -0.0575  [-0.2575, +0.1425]

  ── Historical diagnostics ───────────────────────────────────────────
  History          : 2024-05-07  →  2026-05-06
  Price obs        : 501  |  Return obs: 500
  RV window        : 63 trading days (~3

In [62]:
temp_hist_result = compute_historical_bounds("NVDA", period="2y", v0_window=63)
temp_hist_result

{'initial_guess': [0.1514360944650155,
  1.2562587260160862,
  0.2409139415296975,
  0.43408074360400484,
  -0.05748747513671008],
 'bounds': [(0.10600526612551084, 0.19686692280452014),
  (0.6281293630080431, 3.7687761780482587),
  (0.12045697076484875, 0.481827883059395),
  (0.21704037180200242, 0.8681614872080097),
  (-0.2574874751367101, 0.14251252486328994)],
 'diagnostics': {'n_price_obs': 501,
  'n_return_obs': 500,
  'history_start': '2024-05-07',
  'history_end': '2026-05-06',
  'v0_window_days': 63,
  'rv_latest': 0.1514360944650155,
  'rv_mean': 0.2409139415296975,
  'rv_std': 0.15270743340224913,
  'rv_min': 0.06547422053958293,
  'rv_max': 0.6354050696316165,
  'ar1_b': 0.995014846325333,
  'ar1_implied_theta': 0.1356396657024151,
  'feller_satisfied': True}}

In [63]:
his_asset_price = yf()

TypeError: 'module' object is not callable